In [1]:
import sys
import os
import json

import datasets
from transformers import AutoTokenizer

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.agents.cove_agent import CoVEAgent

### Agent setup

In [2]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

agent = CoVEAgent(
    response_model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    eval_model_name="gemini-2.0-flash", 
    temperature=0.5,
    tokenizer=tokenizer,
)

### SQUAD

In [3]:
dataset_name = "rajpurkar_squad"
path = f"../../../dataset/raw_model_responses/test/test_rajpurkar_squad.json"

data = datasets.load_dataset("json", data_files=path)
data = data["train"]

In [5]:
final_results = []
input_tokens = 0
output_tokens = 0
for sample in data:
    question = sample["additional_info"]["question"]
    context = sample["additional_info"]["context"]
    corrected_responses = []

    for i in range(len(sample["response"])):
        result = agent.model.invoke({
            "initial_question": question,
            "context": context,
        })
        corrected_responses.append(result["corrected_response"] if "corrected_response" in result else result["initial_response"])
        input_tokens += result["input_tokens"]
        output_tokens += result["output_tokens"]

    sample["response"] = corrected_responses
    final_results.append(sample)

2025-10-22 08:54:16.130 | INFO     | src.agents.cove_agent:generate_initial_response:92 - Generated initial response: According to the context, St. John's Cathedral was constructed in the 14th century.
2025-10-22 08:54:17.073 | INFO     | src.agents.cove_agent:generate_questions:120 - Generated questions: ["In what century was St. John's Cathedral constructed?"]
2025-10-22 08:54:17.636 | INFO     | src.agents.cove_agent:answer_questions:152 - Retrieved answers: ["St. John's Cathedral was constructed in the 14th century."]
2025-10-22 08:54:17.637 | INFO     | src.agents.cove_agent:verify_answers:171 - Checking if answer for question: In what century was St. John's Cathedral constructed? is correct
2025-10-22 08:54:17.981 | INFO     | src.agents.cove_agent:verify_answers:191 - Is answer correct: True
2025-10-22 08:54:17.982 | INFO     | src.agents.cove_agent:verify_answers:194 - Answer verification mask: [True]
2025-10-22 08:54:19.748 | INFO     | src.agents.cove_agent:generate_initial_r

In [13]:
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Average input tokens: {input_tokens / (len(final_results)*5)}")
print(f"Average output tokens: {output_tokens / (len(final_results)*5)}")

Input tokens: 283776
Output tokens: 9275
Average input tokens: 2987.1157894736843
Average output tokens: 97.63157894736842


In [9]:
final_results

[{'task_info': {'dataset': 'rajpurkar_squad', 'type': 'Contextual QA'},
  'additional_info': {'answer': ['14th century',
    '14th century',
    '14th century'],
   'context': 'Gothic architecture is represented in the majestic churches but also at the burgher houses and fortifications. The most significant buildings are St. John\'s Cathedral (14th century), the temple is a typical example of the so-called Masovian gothic style, St. Mary\'s Church (1411), a town house of Burbach family (14th century), Gunpowder Tower (after 1379) and the Royal Castle Curia Maior (1407–1410). The most notable examples of Renaissance architecture in the city are the house of Baryczko merchant family (1562), building called "The Negro" (early 17th century) and Salwator tenement (1632). The most interesting examples of mannerist architecture are the Royal Castle (1596–1619) and the Jesuit Church (1609–1626) at Old Town. Among the first structures of the early baroque the most important are St. Hyacinth\'s 

In [10]:
with open(f"../../../dataset/cove_responses/test_cove_{dataset_name}.json", "w") as f:
    json.dump(final_results, f, indent=4)